# LEPset Verification

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from skimage import io

RAW_DATA_DIR = Path("/scratch/data/lepset_raw_20260520/LEPset/labeled")

In [ ]:
# Count the number of images and the number of patients (subdirectories) in the
# "PC" and "NPC" subdirectories
pc_dir = RAW_DATA_DIR / "PC"
num_images_pc = sum(1 for _ in pc_dir.glob("**/*.jpg"))
num_patients_pc = sum(1 for _ in pc_dir.glob("*") if _.is_dir())
print(f"PC: {num_images_pc} images, {num_patients_pc} patients")

npc_dir = RAW_DATA_DIR / "NPC"
num_images_npc = sum(1 for _ in npc_dir.glob("**/*.jpg"))
num_patients_npc = sum(1 for _ in npc_dir.glob("*") if _.is_dir())
print(f"NPC: {num_images_npc} images, {num_patients_npc} patients")

In [ ]:
# Visualize one image per patient for the first 5 PC and NPC patients in a
# 2 x 5 grid. Subdirectories are named {patient} and the images {image}.jpg,
# where {patient} and {images} are integers starting from 0.
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i in range(5):
    patient_dir = pc_dir / str(i)
    image_path = next(patient_dir.glob("*.jpg"))
    img = plt.imread(image_path)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"PC Patient {patient_dir.name}")
    axes[0, i].axis("off")

for i in range(5):
    patient_dir = npc_dir / str(i)
    image_path = next(patient_dir.glob("*.jpg"))
    img = plt.imread(image_path)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"NPC Patient {patient_dir.name}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Compute the height, width, and number of channels statistics for all images
heights = []
widths = []
channels = []

for image_path in RAW_DATA_DIR.glob("**/*.jpg"):
    img = io.imread(image_path)
    heights.append(img.shape[0])
    widths.append(img.shape[1])
    channels.append(img.shape[2] if len(img.shape) == 3 else 1)

print(
    "Height: mean={:.2f}, std={:.2f}, min={}, max={}".format(
        np.mean(heights), np.std(heights), np.min(heights), np.max(heights)
    )
)
print(
    "Width: mean={:.2f}, std={:.2f}, min={}, max={}".format(
        np.mean(widths), np.std(widths), np.min(widths), np.max(widths)
    )
)

n_channels, counts = np.unique(channels, return_counts=True)
print("Channels:")
for n, count in zip(n_channels, counts):
    print(f"  {n} channels: {count} images")

In [ ]:
# Inspect the channels for the first image of PC Patient 2
first_image_path = next((pc_dir / "4").glob("*.jpg"))
img = io.imread(first_image_path)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(img)
axes[0].set_title("Original Image")
axes[0].axis("off")
axes[1].imshow(img[:, :, 0], cmap="gray")
axes[1].set_title("Red Channel")
axes[1].axis("off")
axes[2].imshow(img[:, :, 1], cmap="gray")
axes[2].set_title("Green Channel")
axes[2].axis("off")
axes[3].imshow(img[:, :, 2], cmap="gray")
axes[3].set_title("Blue Channel")
axes[3].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
def generate_mask(sequence: np.ndarray) -> np.ndarray:
    """Generate a boolean mask of active greyscale pixels.

    A pixel is included in the mask if it is greyscale (identical
    value across all channels) in every frame of the sequence, and
    its intensity value changes at least once across the sequence.

    Args:
        sequence: Image sequence array of shape
            (frames, height, width, channels).

    Returns:
        Boolean array of shape (height, width) where True indicates
        a pixel that is greyscale throughout the sequence and whose
        intensity varies across frames.
    """
    # True where all channels are  equal (within 2) to the first channel: (F, H, W)
    grayscale_per_frame = np.all(
        np.abs(sequence.astype(np.int16) - sequence[..., :1].astype(np.int16)) <= 10,
        axis=-1,
    )

    # Pixel must be grayscale in every frame: (H, W)
    always_grayscale = grayscale_per_frame.all(axis=0)

    # Intensity of grayscale pixels is the same on every channel,
    # so channel 0 is sufficient: (F, H, W)
    intensity = sequence[..., 0]

    # Pixel must change intensity by > 5 at least once across frames: (H, W)
    changes_value = np.ptp(intensity, axis=0) > 10
    # changes_value = intensity.max(axis=0) != intensity.min(axis=0)

    mask = always_grayscale & changes_value
    mask = remove_small_holes(mask, connectivity=2, area_threshold=128)

    return mask


example_dir = pc_dir / "0"
image_paths = sorted(example_dir.glob("*.jpg"), key=lambda p: int(p.stem))
sequence = np.stack([io.imread(p) for p in image_paths], axis=0)
mask = generate_mask(sequence)

# Display the the images (top row), generated mask (middle row), and masked
# images (bottom row)
fig, axes = plt.subplots(3, len(image_paths), figsize=(20, 8))
for i in range(len(image_paths)):
    img = io.imread(image_paths[i])
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"PC Patient 0 Image {i}")
    axes[0, i].axis("off")

    axes[1, i].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Mask for Image {i}")
    axes[1, i].axis("off")

    axes[2, i].imshow(img * mask[:, :, np.newaxis])
    axes[2, i].set_title(f"Masked Image {i}")
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Generate scan masks for the cleaned images for the first 5 patients in the PC
# and NPC subdirectories and visualize the results.

from skimage.morphology import (
    binary_closing,
    binary_dilation,
    binary_erosion,
    remove_small_holes,
    remove_small_objects,
)


def estimate_fan_params(mask: np.ndarray) -> dict:
    """Estimate fan parameters from a binary scan mask.

    Assumes the fan apex is at (0, mask.shape[1] // 2) and
    orientation is np.radians(90) (pointing straight down).
    Estimates angle_span from the angular range of boundary
    pixels as seen from the apex.

    Args:
        mask: 2D boolean mask of the scan region.

    Returns:
        Dict with keys: center, radius, angle_span, orientation
        (angle_span and orientation in radians, matching the
        convention used by create_fan_mask).
    """
    apex = (0, mask.shape[1] // 2)

    # Extract boundary pixels
    # Erode several rounds to remove noise before boundary extraction
    cleaned = mask.copy()
    cleaned = remove_small_holes(cleaned)
    for _ in range(10):
        cleaned = binary_erosion(cleaned)
    for _ in range(10):
        cleaned = binary_dilation(cleaned)
    boundary = cleaned & ~binary_erosion(cleaned)
    b_rows, b_cols = np.where(boundary)

    dy = b_rows - apex[0]
    dx = b_cols - apex[1]

    # Discard pixels too close to apex (noise at the tip)
    min_dist = mask.shape[0] * 0.05
    valid = np.sqrt(dx**2 + dy**2) > min_dist

    # Angles from the downward vertical; span = max - min
    angles = np.arctan2(dx[valid], dy[valid])

    return dict(
        center=apex,
        radius=int(np.sqrt(dx[valid] ** 2 + dy[valid] ** 2).max()),
        angle_span=float(angles.max() - angles.min()),
        orientation=np.radians(90),
    )


def create_fan_mask(
    image_shape, center, radius, angle_span, orientation, num_points=100
):
    y_c, x_c = center
    start_angle = orientation - (angle_span / 2)
    end_angle = orientation + (angle_span / 2)

    # Generate arc coordinates
    theta = np.linspace(start_angle, end_angle, num_points)
    x_arc = x_c + radius * np.cos(theta)
    y_arc = y_c + radius * np.sin(theta)

    # Concatenate the origin and arc to create a closed polygon
    poly_x = np.concatenate(([x_c], x_arc))
    poly_y = np.concatenate(([y_c], y_arc))

    # Draw the polygon on an empty mask
    rr, cc = polygon(poly_y, poly_x, image_shape)
    mask = np.zeros(image_shape, dtype=bool)
    mask[rr, cc] = True

    return mask


fig, axes = plt.subplots(3, 5, figsize=(20, 8))
for i in range(5):
    patient_dir = pc_dir / str(i)
    image_paths = sorted(patient_dir.glob("*.jpg"), key=lambda p: int(p.stem))
    images = np.stack([io.imread(p) for p in image_paths], axis=0)
    mask = generate_mask(images)

    axes[0, i].imshow(mask, cmap="gray")
    axes[0, i].set_title(f"PC Patient {patient_dir.name}")
    axes[0, i].axis("off")

    params = estimate_fan_params(mask)
    scan_mask = create_fan_mask(
        # image_shape=mask.shape,
        # center=(0, mask.shape[1] // 2),
        # radius=mask.shape[1] // 2,
        # angle_span=np.radians(120),
        # orientation=np.radians(90),
        image_shape=mask.shape,
        center=(0, mask.shape[1] // 2),
        radius=params["radius"],
        angle_span=params["angle_span"],
        orientation=np.radians(90),
    )

    for j in range(20):
        scan_mask = binary_closing(scan_mask)
    scan_mask = remove_small_holes(mask, connectivity=2, area_threshold=1024)
    scan_mask = remove_small_objects(scan_mask, connectivity=2, min_size=1024)
    # for j in range(5):
    #     scan_mask = binary_erosion(scan_mask)
    # for j in range(5):
    #     scan_mask = binary_dilation(scan_mask)

    # scan_mask = generate_scan_mask(mask)
    axes[1, i].imshow(scan_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Scan Mask for Patient {patient_dir.name}")
    axes[1, i].axis("off")

    # Overlay scan mask on original image
    img = io.imread(image_paths[0])
    axes[2, i].imshow(img)
    axes[2, i].imshow(scan_mask, cmap="gray", alpha=0.5)
    axes[2, i].set_title(f"Overlay for Patient {patient_dir.name}")
    axes[2, i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import math

import cv2
import numpy as np


def create_fan_mask(
    image_shape, center, radius, start_angle, end_angle, num_segments=50
):
    # Create a blank binary mask (all zeros)
    mask = np.zeros(image_shape, dtype=np.uint8)

    # Generate points along the arc of the fan
    pts = [center]
    for i in range(num_segments + 1):
        angle = start_angle + (end_angle - start_angle) * i / num_segments
        angle_rad = math.radians(angle)
        x = int(center[0] + radius * math.cos(angle_rad))
        y = int(center[1] + radius * math.sin(angle_rad))
        pts.append((x, y))

    pts = np.array(pts, np.int32)

    # Draw the filled fan shape (255 for white)
    cv2.fillPoly(mask, [pts], 255)
    return mask


# 1. Initialize your binary image dimensions
height, width = 480, 640

# 2. Define Fan Parameters (center (x,y), radius, start/end angles)
center_pt = (width // 2, height)
fan_radius = 200
start_ang = 210  # e.g., 210 degrees (points left)
end_ang = 330  # e.g., 330 degrees (points right)

# 3. Create the mask
fan_mask = create_fan_mask((height, width), center_pt, fan_radius, start_ang, end_ang)

# 4. Apply to your existing binary image
# Load your binary image here -> binary_img = cv2.imread('binary.png',
# cv2.IMREAD_GRAYSCALE)
# result = cv2.bitwise_and(binary_img, binary_img, mask=fan_mask)

# Save or display
cv2.imwrite("fan_mask.png", fan_mask)

In [ ]:
from skimage.draw import polygon
from skimage.measure import label
from skimage.morphology import binary_closing, disk


def largest_connected_component(mask: np.ndarray) -> np.ndarray:
    """Return a mask containing only the largest connected component.

    Args:
        mask: 2D boolean mask.

    Returns:
        Boolean mask with only the largest component retained.
    """
    labeled = label(mask)
    counts = np.bincount(labeled.flat)
    counts[0] = 0  # ignore background
    return labeled == counts.argmax()


def rowwise_fill(mask: np.ndarray) -> np.ndarray:
    """Fill pixels between leftmost and rightmost set pixel per row.

    Eliminates internal holes (e.g. crosshairs, Doppler overlays)
    without expanding the outer boundary.

    Args:
        mask: 2D boolean mask.

    Returns:
        Boolean mask with each row filled between its outermost
        True pixels.
    """
    filled = np.zeros_like(mask)
    for r in range(mask.shape[0]):
        cols = np.where(mask[r])[0]
        if len(cols) >= 2:
            filled[r, cols[0] : cols[-1] + 1] = True
    return filled


def generate_scan_mask(mask: np.ndarray, border: int = 15) -> np.ndarray:
    """Generate a clean fan-shaped scan mask from a raw pixel mask.

    Pipeline:
        1. Strip a border to remove scale bars / text labels.
        2. Keep the largest connected component.
        3. Binary closing to fill small holes.
        4. Row-wise fill to remove larger internal artifacts.
        5. Intersect with the estimated geometric fan mask.

    Args:
        mask: Raw boolean mask from generate_mask.
        border: Width in pixels of the border strip to zero out.

    Returns:
        Clean boolean mask of the scan region.
    """
    # 1. Exclude image border
    m = mask.copy()
    m[:border, :] = False
    m[-border:, :] = False
    m[:, :border] = False
    m[:, -border:] = False

    # 2. Keep largest connected component
    m = largest_connected_component(m)

    # 3. Binary closing to fill small holes
    m = binary_closing(m, disk(10))

    # 4. Row-wise fill for larger internal artifacts
    m = rowwise_fill(m)

    # 5. Intersect with estimated geometric fan mask
    params = estimate_fan_params(m)
    fan = create_fan_mask(
        image_shape=m.shape,
        center=params["center"],
        radius=params["radius"],
        angle_span=params["angle_span"],
        orientation=params["orientation"],
    )
    return m & fan
    # return fan


# --- Visualise on the first 5 PC patients ---
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

for i in range(5):
    patient_dir = pc_dir / str(i)
    image_paths = sorted(patient_dir.glob("*.jpg"), key=lambda p: int(p.stem))
    images = np.stack([io.imread(p) for p in image_paths], axis=0)
    raw_mask = generate_mask(images)
    scan_mask = generate_scan_mask(raw_mask)

    axes[0, i].imshow(raw_mask, cmap="gray")
    axes[0, i].set_title(f"Raw Mask — PC {patient_dir.name}")
    axes[0, i].axis("off")

    axes[1, i].imshow(scan_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Scan Mask — PC {patient_dir.name}")
    axes[1, i].axis("off")

    img = io.imread(image_paths[0])
    axes[2, i].imshow(img)
    axes[2, i].imshow(scan_mask, cmap="Reds", alpha=0.4)
    axes[2, i].set_title(f"Overlay — PC {patient_dir.name}")
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def create_fan_mask(
    image_shape: tuple,
    center: tuple,
    radius: float,
    angle_span: float,
    orientation: float,
    num_points: int = 100,
) -> np.ndarray:
    """Create a boolean fan-shaped (circular sector) mask.

    Constructs the sector boundary explicitly from its three
    bounding elements: two straight radii and the circular arc,
    directly encoding the convex hull of the sector.

    Args:
        image_shape: (height, width) of the output mask.
        center: (row, col) of the fan apex.
        radius: Radius of the fan in pixels.
        angle_span: Total angular width in radians.
        orientation: Angle of the central axis in radians,
            measured from the positive x-axis.
        num_points: Total points sampled across all three
            boundary segments.

    Returns:
        Boolean mask of shape image_shape with the fan region
        set to True.
    """
    y_c, x_c = center
    theta_start = orientation - angle_span / 2
    theta_end = orientation + angle_span / 2
    n_third = num_points // 3

    # Radius 1: apex → start-angle endpoint
    rad1_x = np.linspace(x_c, x_c + radius * np.cos(theta_start), n_third)
    rad1_y = np.linspace(y_c, y_c + radius * np.sin(theta_start), n_third)

    # Arc: start-angle endpoint → end-angle endpoint
    arc_theta = np.linspace(theta_start, theta_end, num_points - 2 * n_third)
    arc_x = x_c + radius * np.cos(arc_theta)
    arc_y = y_c + radius * np.sin(arc_theta)

    # Radius 2: end-angle endpoint → apex
    rad2_x = np.linspace(x_c + radius * np.cos(theta_end), x_c, n_third)
    rad2_y = np.linspace(y_c + radius * np.sin(theta_end), y_c, n_third)

    poly_x = np.concatenate([rad1_x, arc_x, rad2_x])
    poly_y = np.concatenate([rad1_y, arc_y, rad2_y])

    rr, cc = polygon(poly_y, poly_x, image_shape)
    mask = np.zeros(image_shape, dtype=bool)
    mask[rr, cc] = True

    return mask


# --- Visualise on the first 5 PC patients ---
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

for i in range(5):
    patient_dir = pc_dir / str(i)
    image_paths = sorted(patient_dir.glob("*.jpg"), key=lambda p: int(p.stem))
    images = np.stack([io.imread(p) for p in image_paths], axis=0)
    raw_mask = generate_mask(images)
    scan_mask = generate_scan_mask(raw_mask)
    # params = estimate_fan_params(scan_mask)
    # scan_mask = create_fan_mask(
    #     image_shape=raw_mask.shape,
    #     center=params["center"],
    #     radius=params["radius"],
    #     angle_span=params["angle_span"],
    #     orientation=params["orientation"],
    # )

    # Find the minimum enclosing circle of the scan mask using cv2.minEnclosingCircle
    center, radius = cv2.minEnclosingCircle(scan_mask.astype(np.uint8))
    center = (int(center[1]), int(center[0]))  # convert to (row, col)
    radius = int(radius)
    scan_mask = create_fan_mask(
        image_shape=raw_mask.shape,
        center=center,
        radius=radius,
        angle_span=np.radians(120),
        orientation=np.radians(90),
    )

    axes[0, i].imshow(raw_mask, cmap="gray")
    axes[0, i].set_title(f"Raw Mask — PC {patient_dir.name}")
    axes[0, i].axis("off")

    axes[1, i].imshow(scan_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Scan Mask — PC {patient_dir.name}")
    axes[1, i].axis("off")

    img = io.imread(image_paths[0])
    axes[2, i].imshow(img)
    axes[2, i].imshow(scan_mask, cmap="Reds", alpha=0.4)
    axes[2, i].set_title(f"Overlay — PC {patient_dir.name}")
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2


def estimate_fan_params_mec(mask: np.ndarray) -> dict:
    """Estimate fan parameters using minimum enclosing circle.

    Extracts the outer contour of the mask, estimates angle_span
    from the angular range of contour points viewed from the apex,
    then converts the MEC radius to the true fan radius using the
    geometric relationship for a circular sector.

    For angle_span > π/2 (typical ultrasound fans), the MEC
    diameter equals the chord between the two arc endpoints:
        fan_radius = mec_radius / sin(angle_span / 2)

    For angle_span ≤ π/2, the MEC is the circumcircle of the
    apex and both arc endpoints:
        fan_radius = mec_radius * 2 * cos(angle_span / 2)

    Args:
        mask: Clean boolean mask of the scan region.

    Returns:
        Dict with keys: center, radius, angle_span, orientation
        (angle_span and orientation in radians).
    """
    apex = (0, mask.shape[1] // 2)

    # Extract outer contour — minEnclosingCircle needs points,
    # not a binary image
    contours, _ = cv2.findContours(
        mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
    )
    contour = max(contours, key=cv2.contourArea)
    pts = contour.reshape(-1, 2)  # (N, 2): each row is (col, row)

    # Minimum enclosing circle
    (_, _), mec_radius = cv2.minEnclosingCircle(contour)

    # Angle span from angular range of contour points viewed from apex
    dy = pts[:, 1].astype(float) - apex[0]  # row offset
    dx = pts[:, 0].astype(float) - apex[1]  # col offset
    valid = np.sqrt(dx**2 + dy**2) > mask.shape[0] * 0.05
    angles = np.arctan2(dx[valid], dy[valid])
    angle_span = float(angles.max() - angles.min())

    # Convert MEC radius → fan radius using sector geometry
    if angle_span > np.pi / 2:
        fan_radius = int(np.ceil(mec_radius / np.sin(angle_span / 2)))
    else:
        fan_radius = int(np.ceil(mec_radius * 2 * np.cos(angle_span / 2)))

    return dict(
        center=apex,
        radius=fan_radius,
        angle_span=angle_span,
        orientation=np.radians(90),
    )


def create_fan_mask(
    image_shape: tuple,
    center: tuple,
    radius: float,
    angle_span: float,
    orientation: float,
    num_points: int = 100,
) -> np.ndarray:
    """Create a boolean fan-shaped (circular sector) mask.

    Constructs the sector boundary explicitly from its three
    bounding elements: two straight radii and the circular arc,
    directly encoding the convex hull of the sector.

    Args:
        image_shape: (height, width) of the output mask.
        center: (row, col) of the fan apex.
        radius: Radius of the fan in pixels.
        angle_span: Total angular width in radians.
        orientation: Angle of the central axis in radians,
            measured from the positive x-axis.
        num_points: Total points sampled across all three
            boundary segments.

    Returns:
        Boolean mask of shape image_shape with the fan region
        set to True.
    """
    y_c, x_c = center
    theta_start = orientation - angle_span / 2
    theta_end = orientation + angle_span / 2
    n_third = num_points // 3

    # Radius 1: apex → start-angle endpoint
    rad1_x = np.linspace(x_c, x_c + radius * np.cos(theta_start), n_third)
    rad1_y = np.linspace(y_c, y_c + radius * np.sin(theta_start), n_third)

    # Arc: start-angle endpoint → end-angle endpoint
    arc_theta = np.linspace(theta_start, theta_end, num_points - 2 * n_third)
    arc_x = x_c + radius * np.cos(arc_theta)
    arc_y = y_c + radius * np.sin(arc_theta)

    # Radius 2: end-angle endpoint → apex
    rad2_x = np.linspace(x_c + radius * np.cos(theta_end), x_c, n_third)
    rad2_y = np.linspace(y_c + radius * np.sin(theta_end), y_c, n_third)

    poly_x = np.concatenate([rad1_x, arc_x, rad2_x])
    poly_y = np.concatenate([rad1_y, arc_y, rad2_y])

    rr, cc = polygon(poly_y, poly_x, image_shape)
    mask = np.zeros(image_shape, dtype=bool)
    mask[rr, cc] = True

    return mask


# --- Visualise on the first 5 PC patients ---
fig, axes = plt.subplots(3, 5, figsize=(20, 12))

for i in range(5):
    patient_dir = pc_dir / str(i)
    image_paths = sorted(patient_dir.glob("*.jpg"), key=lambda p: int(p.stem))
    images = np.stack([io.imread(p) for p in image_paths], axis=0)
    raw_mask = generate_mask(images)

    # Clean the raw mask before contour extraction
    scan_mask = generate_scan_mask(raw_mask)

    params = estimate_fan_params_mec(scan_mask)
    scan_mask = create_fan_mask(
        image_shape=raw_mask.shape,
        center=params["center"],
        radius=params["radius"],
        angle_span=params["angle_span"],
        orientation=params["orientation"],
    )

    axes[0, i].imshow(raw_mask, cmap="gray")
    axes[0, i].set_title(f"Raw Mask — PC {patient_dir.name}")
    axes[0, i].axis("off")

    axes[1, i].imshow(scan_mask, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(
        f"PC {patient_dir.name} "
        f"r={params['radius']} "
        f"θ={np.degrees(params['angle_span']):.0f}°"
    )
    axes[1, i].axis("off")

    img = io.imread(image_paths[0])
    axes[2, i].imshow(img)
    axes[2, i].imshow(scan_mask, cmap="Reds", alpha=0.4)
    axes[2, i].set_title(f"Overlay — PC {patient_dir.name}")
    axes[2, i].axis("off")

plt.tight_layout()
plt.show()